<a href="https://colab.research.google.com/github/elhamod/IS834_Fall_2026/blob/main/Session%2009%20-%20Supervised%20Learning%20-%20Model%20Evaluation/Supervised_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IS834 — Session 9**

Work through this notebook in Colab. The lecture cells are filled in and ready to run; the
**Cases** and the **Exercises** are yours to complete — write your code in the `# Your code here` cells and your
reasoning in the `*Provide your answer here*` cells.

Every dataset is read straight from the course repository, so there is nothing to upload or install.

*Adapted from Brock Tibert's IS834 Fall 2025 notebook `10_ML2_Supervised_and_Unsupervised.ipynb`. Edited for Fall 2026 by Mohannad Elhamod.*

# Session 9 — Supervised learning, continued

**Learning goals**

1. Tell a classification problem from a regression problem
2. Judge a classifier honestly: compare accuracy to the base rate, and read a confusion matrix, precision, recall and F1
3. Fit a regression model and judge it with MAE, RMSE and R²
4. Overfitting and underfitting — why a model that scores perfectly on the data it was trained on is usually worthless
5. Compare models on the metric that matches the business question, not the one that looks best


## Setup

In [ ]:
# Everything this notebook needs, imported once at the top.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             precision_score, recall_score, f1_score, mean_absolute_error, root_mean_squared_error, r2_score)

# Show wide tables without pandas truncating them to a handful of columns.
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

sns.set_style("dark")

# Every data file this notebook uses is read straight from the course repository.
# There is nothing to upload: DATA is the folder those files live in.
DATA = "https://raw.githubusercontent.com/elhamod/IS834_Fall_2026/main/data/"


## Classification and regression

Every model in this notebook is *supervised*: it learns from rows where the answer is already known. What changes is the kind of answer.

- **Classification** — the aim is to predict a category. There can be two classes (binary: will this customer subscribe, yes or no?) or more (multiclass).
- **Regression** — we predict a __numeric__ value (what is this house worth?).

Part 1 is classification on the bank data; Part 2 is regression on house prices. The same create / fit / predict recipe from Session 8 works for both — what differs is how you score the result.

![](https://www.springboard.com/blog/wp-content/uploads/2021/10/regression-vs.-classification.png)


## Part 1: Classification — evaluating a model honestly

### Recap: the bank data from Session 8

We are going to use the **bank marketing** dataset — the two files you joined in Session 8. It has a realistic mix of numeric and categorical columns, and a target that is genuinely hard to predict, which makes it a much more honest teaching example than a tidy textbook dataset.


#### Bank marketing data dictionary

| **Attribute** | **Description** | **Type / Values** |
|----------------|-----------------|-------------------|
| **id** | The bank customer's id | String |
| **age** | Age of the client | Numeric |
| **job** | Type of job | Categorical: *admin.*, *unknown*, *unemployed*, *management*, *housemaid*, *entrepreneur*, *student*, *blue-collar*, *self-employed*, *retired*, *technician*, *services* |
| **marital** | Marital status | Categorical: *married*, *divorced*, *single* |
| **education** | Level of education | Categorical: *unknown*, *primary*, *secondary*, *tertiary* |
| **default** | Has credit in default? | Binary: *yes*, *no* |
| **balance** | Average yearly balance (in euros) | Numeric |
| **housing** | Has housing loan? | Binary: *yes*, *no* |
| **loan** | Has personal loan? | Binary: *yes*, *no* |
| **contact** | Contact communication type | Categorical: *unknown*, *telephone*, *cellular* |
| **day** | Last contact day of the month | Numeric |
| **month** | Last contact month of year | Categorical: *jan*, *feb*, ..., *dec* |
| **duration** | Last contact duration (in seconds) | Numeric |
| **campaign** | Number of contacts performed during this campaign (includes last contact) | Numeric |
| **pdays** | Days passed since client was last contacted (-1 = never contacted) | Numeric |
| **previous** | Number of contacts before this campaign | Numeric |
| **poutcome** | Outcome of previous marketing campaign | Categorical: *unknown*, *other*, *failure*, *success* |
| **y** | Client subscribed a term deposit? | Binary: *yes*, *no* |


| **Citation** | Moro et al., 2011 — *Using Data Mining for Bank Direct Marketing: An Application of the CRISP-DM Methodology* ([PDF](http://hdl.handle.net/1822/14838)) |


In [ ]:
# Rebuild Session 8's joined table: one row per customer, with the campaign outcome y attached.
customers = pd.read_csv(DATA + "bank-customers.csv")
marketing = pd.read_csv(DATA + "bank-marketing.csv")

bank = pd.merge(customers, marketing, on="id", how="left", validate="one_to_one")
print(bank.shape)


In [ ]:
# The share of customers who converted -- you computed this in Session 8.
bank["y"].value_counts(normalize=True)


**Question 1.** Only 11.7% of customers converted. Remember that number — why will it matter as soon as we start scoring a model?


**Answer**

*Provide your answer here*

### A first model on the bank data

We start the way Session 8 did — a small decision tree on a couple of columns — and then spend the rest of Part 1 asking whether it is any good.

In [ ]:
# A few random rows, to see what we are actually modelling.
bank.sample(3, random_state=595)


Two columns need converting first. The yes/no `y` column becomes a True/False column called `converted`, True for customers who subscribed, which the scoring functions below need. And a decision tree only works with numeric features, so each job label (`admin.`, `blue-collar`, …) gets an integer code in a new `job_code` column.

In [ ]:
# Target: True if the customer subscribed, False if not.
bank["converted"] = bank["y"] == "yes"

# Features: start deliberately small -- age, plus job as an integer code.
bank["job_code"] = pd.Categorical(bank["job"]).codes

X = bank[["age", "job_code"]]
y = bank["converted"]
X.head(3)


**Splitting the data.** As in Session 8: 80% of the rows to train on, 20% held back to score on. `stratify=y` matters more here than it did with diamonds — it keeps the 11.7% conversion rate in both halves, so the test set has the same base rate as the full data rather than whatever an unlucky shuffle hands you.

In [ ]:
# 80% to train on, 20% held back to score on, with the same conversion rate in both halves.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=595, stratify=y
)
print(X_train.shape, X_test.shape)


In [ ]:
# Fit the tree. The three limits stop it carving the data into tiny, unreliable groups.
model = DecisionTreeClassifier(max_depth=3, min_samples_split=100, min_samples_leaf=50,
                               random_state=595)
model.fit(X_train, y_train)

# Two kinds of output: a hard True/False prediction, and the probability behind it.
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]


In [ ]:
# Draw the tree. Every box is a rule; follow the branches down to reach a prediction.
conversion_labels = ["No", "Yes"]

plt.figure(figsize=(25, 10))
plot_tree(
    model,
    filled=True,
    feature_names=X_train.columns,
    class_names=conversion_labels,
    rounded=True,
    fontsize=10,
)
plt.title("Decision Tree — Predict Marketing Conversion")
plt.show()


### Reading the scorecard

| Metric        | Applies To                | What It Measures                                                              | Explanation                                                                 |
|---------------|---------------------------|--------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| Precision     | Per class, Macro | Of all predicted cases for a class, how many were correct?                    | How careful the model is when making predictions. High precision = fewer false alarms.    |
| Recall        | Per class, Macro | Of all actual cases for a class, how many did the model catch?               | How complete the model is. High recall = few missed cases.                                |
| F1 Score      | Per class, Macro | Harmonic mean of precision and recall                                         | A single score that balances being correct and being complete.                            |
| Support       | Per class                  | Number of actual examples of each class                                       | How many examples we had of each outcome (e.g., converters and non-converters).                        |
| Accuracy      | Overall                    | Total correct predictions / total predictions                                 | How often the model was right overall.                                                    |
| Macro Average | Precision, Recall, F1      | Unweighted average across all classes                                         | Treats all outcome types equally, even rare ones. Simple average.                                      |

Precision vs Recall:
- We might prioritize precision when we are ok missing a few known examples.  I think of this as when the cost of missing those examples is low, and can focus resources accordingly (e.g. Spam detection)
- Recall may be more appropriate when the cost of missing a value is high.  For example, medical screenings.


In [ ]:
# The first five predictions -- a numpy array of True/False values.
print(type(y_pred))
print(y_pred[:5])


In [ ]:
# The matching probabilities of converting. These are what predict() thresholds at 0.5.
print(y_score[:5].round(3))


In [ ]:
# The standard scorecard for a classifier. zero_division=0 silences the warning a never-predicted class triggers.
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
# The confusion matrix behind those numbers: rows are the truth, columns are the prediction.
cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm,
                   index=["actual No", "actual Yes"],
                   columns=["predicted No", "predicted Yes"]))


**Question 2.** The report shows about 88% accuracy. Look at the confusion matrix, then say what this model has actually learned.


**Answer**

*Provide your answer here*

### Improving the model

**In-class exercise.** The model above used only two inputs, age and job, and it found nobody. Change the training process so that the model does better on the test set.

Before you start, decide what "better" should mean. Look back at the metrics table: is overall accuracy the right measure for this business problem?

The three steps below work through one route, changing one thing at a time.


#### Step 1 — stop optimising for accuracy

`class_weight="balanced"` tells the tree to treat one converting customer as worth roughly eight non-converting ones, which is the ratio in the data. The model stops being able to win by saying "no" to everybody.


In [ ]:
# Same two features as before, but the rare class now carries more weight.
balanced_model = DecisionTreeClassifier(
    max_depth=7,
    min_samples_split=100,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=595,
)
balanced_model.fit(X_train, y_train)

balanced_pred = balanced_model.predict(X_test)
print(classification_report(y_test, balanced_pred))


#### Step 2 — turn `job` into dummy variables

`job_code` gave each job an integer, in alphabetical order: `admin.` is 0, `blue-collar` is 1, `entrepreneur` is 2, and so on. Those numbers invent an order that does not exist — nothing makes an entrepreneur "twice" a blue-collar worker — but the model reads the column as a number all the same.

The fix is **dummy variables** (also called one-hot encoding): one yes/no column per category. `job_blue-collar` is `True` for blue-collar customers and `False` for everyone else, and there is one such column for every job. `pd.get_dummies` builds them. `drop_first=True` leaves out one category, because it is already implied when all the others are `False`.

Everything else stays as it was in Step 1: the same two inputs and the same balanced tree. Only the way `job` is written down changes.

In [ ]:
# Same two inputs, with job as one True/False column per category instead of an integer code.
X_dummies = pd.get_dummies(bank[["age", "job"]], columns=["job"], drop_first=True)
X_dummies.head(3)

In [ ]:
# Same split and the same balanced tree as Step 1 -- only the job columns have changed.
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_dummies, y, test_size=0.2, random_state=595, stratify=y
)

dummy_model = DecisionTreeClassifier(
    max_depth=7,
    min_samples_split=100,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=595,
)
dummy_model.fit(Xd_train, yd_train)

dummy_pred = dummy_model.predict(Xd_test)
print(classification_report(yd_test, dummy_pred))

#### Step 3 — give the model more features

Two columns were never going to be enough. Here we use every column in the table: seven numeric ones and nine categorical ones. Each categorical column becomes dummy variables exactly as `job` did in Step 2; `pd.get_dummies` does all nine at once.

In [ ]:
# All 16 columns. get_dummies turns the 9 categorical ones into True/False columns, as in Step 2.
numeric_cols = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
category_cols = ["job", "marital", "education", "default", "housing", "loan",
                 "contact", "month", "poutcome"]

X_full = pd.get_dummies(bank[numeric_cols + category_cols],
                        columns=category_cols, drop_first=True)
print("features before encoding:", len(numeric_cols) + len(category_cols))
print("features after encoding: ", X_full.shape[1])
X_full.head(3)


In [ ]:
# Re-split on the wider feature set. Same random_state, so the same rows land in the same halves.
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_full, y, test_size=0.2, random_state=595, stratify=y
)

full_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=595,
)
full_model.fit(Xf_train, yf_train)

full_pred = full_model.predict(Xf_test)
print(classification_report(yf_test, full_pred))


#### Comparing the models

**One number to compare them: macro F1.** The F1 score combines precision and recall for one class into a single number, and it is only high when both are. The classification report gives an F1 for each class; the **macro** average is the plain, unweighted mean of the two, so the rare "yes" class counts exactly as much as the common "no" class.

That is what makes it a fair way to compare these models. A model that says "no" to everybody scores an F1 of 0 on the "yes" class, and the 88% of easy "no" answers cannot hide that.

In [ ]:
# Compare the four models: overall accuracy, how well they catch converters, and macro F1.
def score_row(name, truth, predicted):
    return {
        "model": name,
        "accuracy": accuracy_score(truth, predicted),
        "precision (yes)": precision_score(truth, predicted, zero_division=0),
        "recall (yes)": recall_score(truth, predicted),
        "F1 (macro)": f1_score(truth, predicted, average="macro"),
    }

comparison = pd.DataFrame([
    score_row("age + job, plain", y_test, y_pred),
    score_row("age + job, balanced", y_test, balanced_pred),
    score_row("age + job dummies, balanced", yd_test, dummy_pred),
    score_row("all features, balanced", yf_test, full_pred),
])
comparison.round(3)

**Question 3.** Look down the table. Which column would you use to pick the best of the four models for this business problem, and which model has the highest accuracy?

**Answer**

*Provide your answer here*

### Overfitting and underfitting

Everything so far has quietly relied on one habit: we scored the model on the **test** set, data it had never seen. This section is about why that habit is not optional.

A model has two ways to fail:

- **Underfitting** — the model is too simple to capture the pattern. It does badly on the training data *and* badly on the test data. A tree with `max_depth=1` asks one question about 45,000 customers.
- **Overfitting** — the model is complex enough to memorise the training data, including its noise and its accidents. It does brilliantly on the training data and badly on the test data. It has learned *this sample*, not the process that generated it.

The gap between the training score and the test score is the diagnostic. A large gap means overfitting; two low scores mean underfitting; two similar, decent scores mean you are somewhere sensible.


#### Two trees, same data, opposite failures

`max_depth` controls how many questions the tree may ask in a row. Nothing else changes between these two fits.


In [ ]:
# Deliberately too simple: one question, then a verdict.
shallow_tree = DecisionTreeClassifier(max_depth=1, random_state=595)
shallow_tree.fit(Xf_train, yf_train)

print("leaves:", shallow_tree.get_n_leaves())
print("train accuracy:", round(accuracy_score(yf_train, shallow_tree.predict(Xf_train)), 4))
print("test  accuracy:", round(accuracy_score(yf_test, shallow_tree.predict(Xf_test)), 4))
print("train F1 (macro):", round(f1_score(yf_train, shallow_tree.predict(Xf_train), average="macro"), 4))
print("test  F1 (macro):", round(f1_score(yf_test, shallow_tree.predict(Xf_test), average="macro"), 4))


In [ ]:
# Deliberately too complex: no depth limit at all, so the tree grows until every leaf is pure.
deep_tree = DecisionTreeClassifier(max_depth=None, random_state=595)
deep_tree.fit(Xf_train, yf_train)

print("leaves:", deep_tree.get_n_leaves())
print("train accuracy:", round(accuracy_score(yf_train, deep_tree.predict(Xf_train)), 4))
print("test  accuracy:", round(accuracy_score(yf_test, deep_tree.predict(Xf_test)), 4))
print("train F1 (macro):", round(f1_score(yf_train, deep_tree.predict(Xf_train), average="macro"), 4))
print("test  F1 (macro):", round(f1_score(yf_test, deep_tree.predict(Xf_test), average="macro"), 4))


**Question 4.** The deep tree scores 1.000 on the training data. Why is that not cause for celebration?


**Answer**

*Provide your answer here*

#### Watching the gap open up

Fitting the same tree at a range of depths and plotting both scores together is the standard way to see this. The training curve keeps climbing. The test curve climbs and then levels off — and once it stops improving, every extra level of depth is complexity that buys nothing on new data.


In [ ]:
# Fit one tree per depth and record how it scores on both halves of the data.
depths = [1, 2, 3, 4, 5, 6, 8, 10, 12, 16, 20, 25]
rows = []

for depth in depths:
    trial = DecisionTreeClassifier(max_depth=depth, random_state=595)
    trial.fit(Xf_train, yf_train)
    rows.append({
        "max_depth": depth,
        "leaves": trial.get_n_leaves(),
        "train_f1": f1_score(yf_train, trial.predict(Xf_train), average="macro"),
        "test_f1": f1_score(yf_test, trial.predict(Xf_test), average="macro"),
    })

curve = pd.DataFrame(rows)
curve.round(3)


In [ ]:
# Plot the two curves together -- the gap between them IS the overfitting.
plt.figure(figsize=(8, 5))
plt.plot(curve["max_depth"], curve["train_f1"], marker="o", label="train")
plt.plot(curve["max_depth"], curve["test_f1"], marker="o", label="test")
plt.axvline(curve.loc[curve["test_f1"].idxmax(), "max_depth"],
            color="grey", linestyle="--", label="best test score")
plt.xlabel("max_depth (model complexity)")
plt.ylabel("F1 (macro)")
plt.title("Train vs test as the tree gets deeper")
plt.legend()
plt.show()

best = curve.loc[curve["test_f1"].idxmax()]
print(f"Best test F1 (macro) {best['test_f1']:.3f} at max_depth={int(best['max_depth'])} "
      f"({int(best['leaves'])} leaves)")


**Question 5.** Reading the plot: which region is underfitting, which is overfitting, and where would you stop?


**Answer**

*Provide your answer here*

> **One honest caveat.** We have now used the test set to *choose* `max_depth`, which means the test score is a little optimistic — we picked the depth that happened to look best on those particular rows. Done properly, you split three ways (train / validation / test) or use cross-validation to choose the setting, and keep a genuinely untouched test set for the final number. That is beyond this course, but knowing the caveat exists is not.


### Other classifiers in scikit-learn

There are all sorts of classifiers that exist right inside scikit-learn!

- [KNN Classifier/Regressor](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
    - Use "similar" records to make predictions.  This can be used for Regression or Classification Problems.

- [RandomForest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
    - A collection of Decision Trees. Uses "wisdom of the crowds" to generate predictions.
- [Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)
    - A linear model that uses the probability of class assignment to learn the model.  Typically for binary problems, but there are adaptations to support multi-class problems via one-vs-the-rest.
- [Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html#classification)
    - Creates boundaries __across the full feature space__ to separate the categories/classes in the dataset.
- [Gradient Boosting Trees](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosted-trees)
    - A collection of small decision trees, but instead of combining the results, trees are fit __sequentially__ in order to try to improve on the mistakes (errors) of the previous. The errors are used as the target to narrow the focus.


__Random Forest__:

![](https://www.spotfire.com/content/dam/spotfire/images/graphics/inforgraphics/random-forest-diagram.svg)

The process uses random samples (with replacement) for selecting records for the tree, as well as for each split consideration.

![](https://algobeans.com/wp-content/uploads/2016/08/bagging.gif)

__Additional Notes__:

- Scikit-Learn has a number of tools that can be used to clean and adjust data.  This notebook uses pandas for that, but scikit-learn can also be used to encode categorical data, adjust numeric inputs, etc.
- Scikit learn has the concept of a pipeline, where the steps above can be combined into a single object.  This is a convenient way of strongly coupling all of the tasks together, which can help with downstream tasks.


### Case: predicting diabetes risk from a health survey

**In-class exercise.** You now have the whole classification toolkit: a baseline, a balanced tree, and the metrics to judge it. Take some time to explore this dataset and see how well various models fit it.

**Source**
Kaggle: https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset
Original: https://www.cdc.gov/pcd/issues/2019/19_0109.htm

**Problem context**

**Background.** A regional healthcare network is working to improve outcomes for patients with chronic diseases, with a major focus on type II diabetes, one of the most costly and widespread health issues in the United States. With over 34 million Americans affected and another 88 million at risk, the system recognizes that early identification and intervention are crucial—not just for improving health outcomes, but also for reducing long-term healthcare costs.

**Problem.** Many individuals living with prediabetes or diabetes are unaware of their condition. This lack of awareness delays intervention and can lead to serious complications such as heart disease, kidney failure, and amputations—driving up both human and financial costs.

The healthcare network has access to a large volume of self-reported health survey data (from the CDC’s 2015 BRFSS dataset) on patient behaviors, health history, and lifestyle factors. The leadership team wants to use machine learning to help predict which individuals are most at risk of having undiagnosed prediabetes or diabetes, allowing them to target outreach programs more effectively.

**Goal.** Use the `medical-tests.csv` dataset to train and evaluate a machine learning model that attempts to identify the label associated with the column `Diabetes_012`.

**Data dictionary**

- **Diabetes_012**: Target variable  
  - 0 = No diabetes or only during pregnancy  
  - 1 = Prediabetes  
  - 2 = Diabetes  

- **HighBP**: High blood pressure diagnosis  
  - 0 = No  
  - 1 = Yes  

- **HighChol**: High cholesterol diagnosis  
  - 0 = No  
  - 1 = Yes  

- **CholCheck**: Had a cholesterol check in the past 5 years  
  - 0 = No  
  - 1 = Yes  

- **BMI**: Body Mass Index (numeric)

- **Smoker**: Smoked at least 100 cigarettes in lifetime  
  - 0 = No  
  - 1 = Yes  

- **Stroke**: Ever told had a stroke  
  - 0 = No  
  - 1 = Yes  

- **HeartDiseaseorAttack**: Ever diagnosed with coronary heart disease (CHD) or myocardial infarction (MI)  
  - 0 = No  
  - 1 = Yes  

- **PhysActivity**: Participated in physical activity (past 30 days, excluding job-related activity)  
  - 0 = No  
  - 1 = Yes  

- **Fruits**: Consumes fruit 1 or more times per day  
  - 0 = No  
  - 1 = Yes  

- **Veggies**: Consumes vegetables 1 or more times per day  
  - 0 = No  
  - 1 = Yes  

- **HvyAlcoholConsump**: Heavy alcohol consumption  
  - 0 = No  
  - 1 = Yes  
  - (Defined as: Men >14 drinks/week, Women >7 drinks/week)  

- **AnyHealthcare**: Has any form of health care coverage  
  - 0 = No  
  - 1 = Yes  

- **NoDocbcCost**: Needed to see a doctor in the past 12 months but couldn’t due to cost  
  - 0 = No  
  - 1 = Yes  

- **GenHlth**: General health status  
  - 1 = Excellent  
  - 2 = Very Good  
  - 3 = Good  
  - 4 = Fair  
  - 5 = Poor  

- **MentHlth**: Number of days in the past 30 days with poor mental health (0–30 scale)

- **PhysHlth**: Number of days in the past 30 days with poor physical health (0–30 scale)

- **DiffWalk**: Serious difficulty walking or climbing stairs  
  - 0 = No  
  - 1 = Yes  

- **Sex**: Biological sex  
  - 0 = Female  
  - 1 = Male  

- **Age**: Age group (13 levels)  
  - 1 = 18–24  
  - 2 = 25–29  
  - 3 = 30–34  
  - 4 = 35–39  
  - 5 = 40–44  
  - 6 = 45–49  
  - 7 = 50–54  
  - 8 = 55–59  
  - 9 = 60–64  
  - 10 = 65–69  
  - 11 = 70–74  
  - 12 = 75–79  
  - 13 = 80 or older  

- **Education**: Education level  
  - 1 = Never attended school or only kindergarten  
  - 2 = Grades 1–8  
  - 3 = Grades 9–11  
  - 4 = High school graduate  
  - 5 = Some college or technical school  
  - 6 = College graduate  

- **Income**: Income level  
  - 1 = Less than $10,000  
  - 2 = $10,000–$15,000  
  - 3 = $15,000–$20,000  
  - 4 = $20,000–$25,000  
  - 5 = $25,000–$35,000  
  - 6 = $35,000–$50,000  
  - 7 = $50,000–$75,000  
  - 8 = $75,000 or more

In [ ]:
# Your code here -- load medical-tests.csv and look at how common each Diabetes_012 class is

Two choices before fitting. The full file has 253,680 rows, which makes every fit slow; a random 50,000-row sample behaves the same way and keeps the notebook responsive. And prediabetes (1) and diabetes (2) are both worth a screening call, so we combine them into one True/False column, `at_risk` — which also deals with prediabetes being only 1.8% of the rows on its own.

In [ ]:
# Your code here -- take a 50,000-row sample and add a True/False at_risk column

In [ ]:
# Your code here -- split, fit a classifier, and print the classification report

In [ ]:
# Your code here -- which answers did the model rely on most?

**Question 6.** How well does the model find the people at risk, and which metric would you show the leadership team?

**Answer**

*Provide your answer here*

### Follow up Questions and Exercises

**Question 1.** A colleague reports a churn model with 97% accuracy. What is the first question you ask?


**Answer**

*Provide your answer here*

**Question 2.** The diamonds model in Session 8 and the first bank model in this notebook both scored roughly 88%. Why is one of them a good result and the other one useless?

**Answer**

*Provide your answer here*

#### Exercise 1

Fit a decision tree that predicts conversion using only `duration` (the length of the last phone call). Score it, then explain why you should not deploy it.


In [ ]:
# Your code here

**Answer**

*Provide your answer here*

#### Exercise 2

The bank can only afford to call 10% of the customers in the test set. Refit the all-features model without `duration` (Exercise 1 shows why it cannot be used), use its predicted probabilities to pick the best 10%, and work out what share of the actual converters that call list would reach.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

## Part 2: Regression

A regression model predicts a number, so it cannot be "right" or "wrong" the way a yes/no call is — every prediction misses by *some* amount. The metrics below measure how big those misses are.

### Regression metrics — understanding model fit and risk

| **Metric** | **Full Name** | **Intuition** | **Units of Observation** | **Good Value?** | **Risk Interpretation** |
|-------------|---------------|----------------|----------------------------|------------------|--------------------------|
| **MAE** | Mean Absolute Error | Average size of prediction errors, ignoring direction. | Same units as the target (e.g., dollars, sales, prices). | Lower is better | Shows typical prediction error magnitude. High MAE = consistent everyday misses. |
| **RMSE** | Root Mean Squared Error | Squares each error, averages them, then takes the square root to return to the target’s units. | Same units as the target. | Lower is better | Highlights how far predictions typically are from actuals, punishing big misses more than MAE. |
| **R²** | Coefficient of Determination | Fraction of variation in the target explained by the model. | Unitless (0–1 scale). | Closer to 1 is better | High R² can overstate performance if the model is overfitting; test on new data. |
| **MAPE** | Mean Absolute Percentage Error | Average percentage difference between predictions and actuals. | Percentage (%) | Lower is better | Easy to interpret, but unreliable when true values ≈ 0; small denominators can distort risk. |


### Some common regression models

Linear Regression
- Likely have seen this in your stats class
- Linear model, can be used to estimate the effects of a feature on a target/outcome


Trees
- Just like the approach we saw in Session 8 and in Part 1, Decision trees can be framed for a Regression task
- The leaf/terminal node takes the average of the records and that is the prediction for a given "rule"
- Random Forest Regression is a collection of trees, shown above


KNN Regressor
- Discussed Above.  We use `N` similar observations and take the average of those target values to become the "prediction".
- Similarity is defined based on the features used.

More Advanced
- Neural Network regression
- Feed Forward Neural Network, foundation of Deep Learning


### Case: predicting California house values

We are going to use a housing dataset that attempts to predict the median home value for the geographic unit.

Data dictionary: https://docs.google.com/document/u/1/d/e/2PACX-1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia_DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN/pub


In [ ]:
# The California housing data: one row per census block group.
housing = pd.read_csv(DATA + "california_housing.csv")
housing.head()


#### Explore the data

**In-class exercise.**

1. Review the summary statistics
2. Review the correlation matrix
3. Create a scatterplot with the variable that is most correlated to median home value. What do you notice about the plot?


In [ ]:
# Your code here -- 1) summary statistics

In [ ]:
# Your code here -- 2) correlation matrix

In [ ]:
# Your code here -- 3) scatterplot against the most correlated variable

**Question 1.** Which variable is most strongly correlated with median house value, and what do you notice in the scatterplot?

**Answer**

*Provide your answer here*

#### Fit and compare two models

**In-class exercise.**

- Split the data into training and test sets
- Fit a Linear Regression and a Decision Tree to predict the median house value (target)
- Which one is better?
- For your test set, create a scatter plot of the predicted values (y) against the known value (x)
- Tricky: generate a prediction for the example record shown below

| **Feature** | **Value** |
|--------------|-----------|
| longitude | -120.87 |
| latitude | 37.77 |
| housing_median_age | 9.0 |
| total_rooms | 4838.0 |
| total_bedrooms | 920.0 |
| population | 2460.0 |
| households | 923.0 |
| median_income | 3.5959 |


In [ ]:
# Your code here -- split the data into training and test sets

In [ ]:
# Your code here -- fit a Linear Regression and a Decision Tree

In [ ]:
# Your code here -- score both models on the training and the test set (R², MAE, RMSE)

**Question 2.** Which model is better?


**Answer**

*Provide your answer here*

#### Limit the tree's depth

In [ ]:
# Your code here -- refit the tree with a depth limit: try several max_depth values and keep the best on the test set

**Question 3.** Compare the depth-limited tree with the other two models, on the training data and on the test data. What changed?

**Answer**

*Provide your answer here*

#### Predicted vs actual

In [ ]:
# Your code here -- scatter plot of predicted values (y) against actual values (x) on the test set

**Question 4.** What does the shape of that scatter tell you that the R² does not?


**Answer**

*Provide your answer here*

#### Predict one new house

In [ ]:
# Your code here -- predict the example record with each model

**Question 5.** The three models give three different prices for the same house. How should you report this prediction?

**Answer**

*Provide your answer here*

### Follow up Questions and Exercises

#### Exercise 1

Take the linear regression fitted above and look at its coefficients. What does the `median_income` coefficient say, and what is the catch?


In [ ]:
# Your code here

**Answer**

*Provide your answer here*

## What comes next

Every model in this notebook learned from a column of known answers — `converted`, `at_risk`, `median_house_value`. Session 10 takes that column away. Clustering looks for groups in the rows without being told what the groups should be, which raises a harder question than any metric here: how do you judge a model when there is no right answer to score it against?